# Task 21 - Tuning the random forest

**Owner:** Viviana Pajic. Some implementations and initial explanations were prepared with AI assistance; see [the assistance record](../docs/AI_USE.md).

The random forest is our best model so far (average precision 0.783), but it
overfits: its training score (0.95) is far above its validation score. We tune
the two settings that control this. `min_samples_leaf` is the smallest number
of mushrooms a leaf may contain; bigger leaves stop the trees from memorising
single training rows. `max_features` is how many features each split may look
at; fewer features make the trees more different from each other. The number
of trees stays at 150, as in the baseline, so only these two settings change.

## Method: nested cross-validation

As for logistic regression (task 20), the search runs inside each of the 10
outer folds: a grid search with 3 inner folds chooses the settings using only
the outer training rows, and the outer validation fold scores that choice. The
result is therefore directly comparable with the baseline in notebook 02, and
the reserved test rows are never used. This cell runs about 360 forests and
takes a few minutes.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src/mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from mushroom_workflow import REPORTS, candidates, evaluate_model, load_development

X, y, _ = load_development()
baseline = pd.read_csv(REPORTS / 'baseline_cv.csv').set_index('model')
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print('Development rows:', len(X), '| baseline RF AP: %.4f' % baseline.loc['random_forest', 'cv_average_precision'])

Development rows: 4000 | baseline RF AP: 0.7831


In [3]:
grid = {'model__min_samples_leaf': [1, 3, 5, 10],
        'model__max_features': ['sqrt', 0.3, 0.5]}
search = GridSearchCV(candidates(X)['random_forest'], grid,
                      scoring='average_precision', cv=inner_cv, n_jobs=1)
summary, folds, _ = evaluate_model('random_forest_tuned', search, X, y)
pd.DataFrame([summary]).to_csv(REPORTS / 'tuned_random_forest_cv.csv', index=False)
folds.to_csv(REPORTS / 'tuned_random_forest_folds.csv', index=False)
comparison = pd.DataFrame([baseline.loc['random_forest'], pd.Series(summary)],
                          index=['baseline', 'tuned'])
display(comparison[['cv_average_precision', 'cv_average_precision_std',
                    'cv_roc_auc', 'train_average_precision']].astype(float).round(4))

,cv_average_precision,cv_average_precision_std,cv_roc_auc,train_average_precision
baseline,0.7831,0.0087,0.8334,0.9507
tuned,0.7885,0.0109,0.8412,1.0000


## Which settings does the search prefer?

We run the search once on all 4,000 development rows to see which settings it
chooses and how close the alternatives are.

In [4]:
search.fit(X, y)
results = pd.DataFrame(search.cv_results_)[['param_model__min_samples_leaf', 'param_model__max_features',
                                           'mean_test_score', 'std_test_score', 'rank_test_score']]
display(results.sort_values('rank_test_score').head(6).round(4))
print('Chosen settings:', search.best_params_)

,param_model__min_samples_leaf,param_model__max_features,mean_test_score,std_test_score,rank_test_score
5,3,0.3,0.7787,0.0132,1
4,1,0.3,0.7781,0.0157,2
9,3,0.5,0.7772,0.0105,3
8,1,0.5,0.7733,0.0175,4
0,1,sqrt,0.7716,0.0194,5
6,5,0.3,0.7705,0.0091,6


Chosen settings: {'model__max_features': 0.3, 'model__min_samples_leaf': 3}


In [5]:
base_folds = pd.read_csv(REPORTS / 'baseline_folds.csv').query("model == 'random_forest'").set_index('fold')
tuned_folds = folds.set_index('fold')
paired = pd.DataFrame({'baseline_ap': base_folds['average_precision'],
                       'tuned_ap': tuned_folds['average_precision']})
paired['difference'] = paired['tuned_ap'] - paired['baseline_ap']
display(paired.round(4))
print('Folds where tuning helps: %d of %d' % ((paired['difference'] > 0).sum(), len(paired)))
print('Mean difference: %.4f' % paired['difference'].mean())

,baseline_ap,tuned_ap,difference
fold,,,
0,0.7808,0.7837,0.0029
1,0.7920,0.7918,-0.0002
2,0.7797,0.7865,0.0067
3,0.7744,0.7973,0.0229
4,0.7821,0.8000,0.0179
5,0.7804,0.7901,0.0097
6,0.7768,0.7829,0.0061
7,0.7794,0.7710,-0.0084
8,0.8057,0.8078,0.0021


Folds where tuning helps: 7 of 10
Mean difference: 0.0054


## What this completes

Tuning improves the random forest slightly: average precision rises from 0.7831
to 0.7885 and ROC-AUC from 0.8334 to 0.8412, and the tuned forest is ahead in
7 of 10 identical folds. The gain comes from `max_features = 0.3` (about 19 of
the 64 prepared columns per split instead of about 8 with `sqrt`): after one-hot
encoding most columns are sparse and weak, so looking at more of them per split
finds useful splits more often.

Larger leaves did not help, which contradicts our starting idea that limiting
overfitting would improve the forest. The outer searches mostly chose fully
grown trees, so the training score rose to 1.00 while validation still improved.
For a random forest, averaging many different deep trees generalises even when
each tree memorises its sample, so the validation score, not the training gap,
is what counts. The gain is smaller than the fold-to-fold spread, so we treat
the tuned forest as a modest improvement and the leading candidate for task 28.